# Deploy_CU_Merger_Demo: one-click installer

This notebook builds the **Contoso CU + Fabrikam CU merger demo** in your Fabric tenant. You set the parameters and select **Run all**. It's safe to rerun: anything that already exists is reused or updated.

| Step | What it builds | Where |
|---|---|---|
| 1 | Workspaces **1-DEV** (plus **2-TEST-QA** and **3-PROD** if you ask for them), on your capacity | New workspaces |
| 2 | Lakehouses **Contoso_CU**, **Fabrikam_CU**, **Enterprise_Ref**, loaded by the `Load_Source_Data` notebook | Each stage |
| 3 | Warehouse **Combined-Data**: schema, `usp_Load_Combined`, the load and the validation queries | Each stage |
| 4 | Semantic model and report **Monthly Member Intelligence Hub**, bound to that stage's warehouse, with credentials set and a first refresh | Each stage |
| 5 | Personas: workspace access (directors and executive as Viewer) and **Security Scope** RLS role membership | Each stage |
| 6 | Optional deployment pipeline across Dev, Test and Prod | Tenant |
| 7 | Real-Time Estate Monitor: **Monitor_LH**, eventhouse, KQL views, dashboard, eventstream, scheduled collector, simulator and `Fabric_Governance` | **This** workspace |

**Before you run it**
1. Create an admin workspace (for example **0-MONITOR**) on a paid Fabric capacity, and import this notebook into it. The Estate Monitor is built here, and the demo workspaces are put on the same capacity.
2. Push the kit to a **public** GitHub repo and set `REPO_RAW_ROOT`. Or use `REPO_SOURCE = "files"`: attach a lakehouse to this notebook and upload the whole kit folder to its `Files` area.
3. Create the persona accounts in Entra ID (`west_director@<your domain>` and so on, see doc 01 section 6). The installer grants them access but does not create users.
4. Run it as a **Fabric administrator** who is also **capacity admin** (or contributor) on the capacity. Your Fabric tenant settings must allow creating workspaces and using Copilot.

**Still manual afterwards.** At the end the notebook prints the short list: Fabric event sources on the eventstream, the org app, endorsement and Copilot settings, and alerts.

## Parameters

In [ ]:
# ---------------- Required ----------------
TENANT_DOMAIN = "yourtenant.onmicrosoft.com"     # domain of the persona accounts (west_director@<this>, ...)

# Where the kit lives
REPO_SOURCE   = "github"                          # "github" (public repo) or "files" (kit uploaded to the attached lakehouse)
REPO_RAW_ROOT = "https://raw.githubusercontent.com/<github-user>/<repo-name>/main"   # repo ROOT, not /data
FILES_ROOT    = "/lakehouse/default/Files/fabric-cu-merger-demo"                     # REPO_SOURCE = "files"

# ---------------- Capacity and workspaces ----------------
CAPACITY_ID    = ""                               # blank = the capacity of THIS workspace
STAGES         = {"Dev": "1-DEV", "Test": "2-TEST-QA", "Prod": "3-PROD"}
BUILD_STAGES   = ["Dev"]                          # e.g. ["Dev", "Test", "Prod"] for the pipeline story in Act 2
CREATE_PIPELINE = False                           # needs all three stages
PIPELINE_NAME  = "CU Merger Demo"

# ---------------- Core demo ----------------
BUILD_CORE         = True
WAREHOUSE_NAME     = "Combined-Data"
MODEL_NAME         = "Monthly Member Intelligence Hub"
REPORT_NAME        = "Monthly Member Intelligence Hub"
RLS_ROLE           = "Security Scope"
SET_CREDENTIALS    = True                         # OAuth2 credentials on the model's warehouse data source
REFRESH_MODEL      = True
ADD_RLS_MEMBERS    = True                         # needs XMLA read-write on the capacity (default on Fabric capacities)
GRANT_WORKSPACE_ACCESS = True

# persona prefix -> workspace role. Directors and the executive must be Viewers (higher roles bypass RLS).
PERSONAS = {
    "mgmt_analyst":   "Contributor",   # Dev only
    "west_director":  "Viewer",
    "east_director":  "Viewer",
    "north_director": "Viewer",
    "south_director": "Viewer",
    "retail_exec":    "Viewer",
}
DEV_ONLY_PERSONAS = ["mgmt_analyst"]

# ---------------- Real-Time Estate Monitor (built in THIS workspace) ----------------
BUILD_ESTATE_MONITOR = True
MONITOR_LAKEHOUSE    = "Monitor_LH"
CREATE_EVENTSTREAM   = True
EVENTSTREAM_NAME     = "EstateMonitor_ES"
EVENTHOUSE_NAME      = "EstateMonitor"
# How the collector and simulator get the eventstream connection string:
KEY_VAULT_URI        = ""                         # set to store it as a Key Vault secret (you need Set permission on secrets)
EVENTSTREAM_SECRET_NAME = "estate-monitor-eventstream"
ALLOW_INLINE_CONNECTION_STRING = True             # no Key Vault: write it into the two notebooks (fine for a demo tenant)
SCHEDULE_COLLECTOR   = True                       # hourly; each run polls for 55 minutes
START_COLLECTOR_NOW  = True                       # first run now, so the baseline inventory lands right away
IMPORT_GOVERNANCE    = True

## Helpers

In [ ]:
import base64, json, os, re, time, datetime as dt, requests, pandas as pd
import sempy.fabric as fabric

try:
    nbu = notebookutils
except NameError:
    from notebookutils import mssparkutils as nbu

assert "yourtenant" not in TENANT_DOMAIN, "Set TENANT_DOMAIN first."
if REPO_SOURCE == "github":
    assert "<github-user>" not in REPO_RAW_ROOT, "Set REPO_RAW_ROOT to your repo, or use REPO_SOURCE = 'files'."

FABRIC_API = "https://api.fabric.microsoft.com/v1/"
PBI_API    = "https://api.powerbi.com/v1.0/myorg/"
THIS_WS    = fabric.get_workspace_id()
LOG = []                                    # (stage, step, status, detail)

def log(stage, step, status, detail=""):
    LOG.append((stage, step, status, str(detail)[:300]))
    print(f"[{stage}] {step}: {status} {detail if detail else ''}")

def token(aud="pbi"):
    return nbu.credentials.getToken(aud)

def call(method, url, base=FABRIC_API, ok=(200, 201, 202), **kw):
    full = url if url.startswith("http") else base + url.lstrip("/")
    for attempt in range(8):
        r = requests.request(method, full, headers={"Authorization": f"Bearer {token()}"}, timeout=120, **kw)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(min(int(r.headers.get("Retry-After", 5 * (attempt + 1))), 60)); continue
        break
    if ok and r.status_code not in ok:
        raise RuntimeError(f"{method} {full} -> HTTP {r.status_code}: {r.text[:600]}")
    return r

def lro(r):
    """Wait for a Fabric long-running operation. Returns the result JSON when there is one."""
    if r.status_code != 202:
        return r.json() if r.content else {}
    loc = r.headers.get("Location")
    while True:
        time.sleep(int(r.headers.get("Retry-After", 3)))
        r = call("GET", loc, ok=None)
        st = (r.json() if r.content else {}).get("status")
        if st == "Succeeded":
            res = call("GET", loc.rstrip("/") + "/result", ok=None)
            return res.json() if res.status_code == 200 and res.content else {}
        if st in ("Failed", "Cancelled"):
            raise RuntimeError(f"Operation {st}: {r.text[:600]}")

def paged(url):
    out = []
    while url:
        j = call("GET", url).json()
        out += j.get("value", [])
        url = j.get("continuationUri")
    return out

# ---------- repo access ----------
def _gh():
    m = re.match(r"https://raw\.githubusercontent\.com/([^/]+)/([^/]+)/([^/]+)/?(.*)$", REPO_RAW_ROOT.rstrip("/"))
    if not m: raise ValueError("REPO_RAW_ROOT must look like https://raw.githubusercontent.com/<user>/<repo>/<branch>")
    return m.groups()

_TREE = None
def repo_list(prefix):
    """All file paths in the kit under prefix (relative to the kit root)."""
    global _TREE
    if REPO_SOURCE == "files":
        root = FILES_ROOT.rstrip("/"); base = f"{root}/{prefix}".rstrip("/")
        return sorted(os.path.relpath(os.path.join(d, f), root).replace(os.sep, "/")
                      for d, _, fs in os.walk(base) for f in fs)
    if _TREE is None:
        user, repo, branch, sub = _gh()
        t = requests.get(f"https://api.github.com/repos/{user}/{repo}/git/trees/{branch}?recursive=1", timeout=60)
        t.raise_for_status()
        pre = (sub + "/") if sub else ""
        _TREE = [x["path"][len(pre):] for x in t.json()["tree"] if x["type"] == "blob" and x["path"].startswith(pre)]
    return sorted(p for p in _TREE if p.startswith(prefix))

def repo_bytes(rel):
    if REPO_SOURCE == "files":
        with open(f"{FILES_ROOT.rstrip('/')}/{rel}", "rb") as f: return f.read()
    from urllib.parse import quote
    r = requests.get(f"{REPO_RAW_ROOT.rstrip('/')}/{quote(rel)}", timeout=60); r.raise_for_status()
    return r.content

def repo_text(rel): return repo_bytes(rel).decode("utf-8-sig")
def b64(data): return base64.b64encode(data if isinstance(data, bytes) else data.encode("utf-8")).decode()

# ---------- workspaces and items ----------
def capacity_id():
    if CAPACITY_ID: return CAPACITY_ID
    cid = call("GET", f"workspaces/{THIS_WS}").json().get("capacityId")
    assert cid, "This workspace isn't on a capacity. Set CAPACITY_ID."
    return cid

def ensure_workspace(name, cap):
    ws = next((w for w in paged("workspaces") if w["displayName"] == name), None)
    if ws is None:
        ws = call("POST", "workspaces", json={"displayName": name, "capacityId": cap}).json()
        log(name, "workspace", "created")
    else:
        if ws.get("capacityId") != cap:
            lro(call("POST", f"workspaces/{ws['id']}/assignToCapacity", json={"capacityId": cap}))
        log(name, "workspace", "exists")
    return ws["id"]

def find_item(ws, kind, name):
    return next((i for i in paged(f"workspaces/{ws}/items?type={kind}") if i["displayName"] == name), None)

def ensure_item(ws, kind, name, definition=None, update=True, stage=""):
    it = find_item(ws, kind, name)
    if it is None:
        body = {"displayName": name, "type": kind}
        if definition: body["definition"] = definition
        lro(call("POST", f"workspaces/{ws}/items", json=body))
        for _ in range(30):
            it = find_item(ws, kind, name)
            if it: break
            time.sleep(5)
        log(stage, f"{kind} {name}", "created")
    elif definition and update:
        lro(call("POST", f"workspaces/{ws}/items/{it['id']}/updateDefinition", json={"definition": definition}))
        log(stage, f"{kind} {name}", "updated")
    else:
        log(stage, f"{kind} {name}", "exists")
    return it

def wait_props(ws, path, key, tries=60):
    for _ in range(tries):
        p = call("GET", f"workspaces/{ws}/{path}").json().get("properties") or {}
        v = p
        for k in key.split("."): v = (v or {}).get(k)
        if v: return p
        time.sleep(10)
    raise TimeoutError(f"{path}: {key} not ready")

# ---------- notebooks ----------
def set_params(src, values):
    for var, val in values.items():
        pat = re.compile(rf'^({re.escape(var)}\s*=\s*)("[^"\n]*"|\'[^\'\n]*\'|[^#\n]*?)(\s*#.*)?$', re.M)
        lit = json.dumps(val) if isinstance(val, str) else repr(val)
        src, n = pat.subn(lambda m: m.group(1) + lit + (m.group(3) or ""), src, count=1)
        if n == 0:
            print(f"  note: parameter {var} not found, left as is")
    return src

def notebook_definition(rel, params=None, lakehouse=None):
    nb = json.loads(repo_text(rel))
    done = False
    for c in nb["cells"]:
        if c["cell_type"] == "code" and not done and params:
            c["source"] = set_params("".join(c["source"]), params).splitlines(True); done = True
    if lakehouse:
        ws_id, lh_id, lh_name = lakehouse
        nb.setdefault("metadata", {}).setdefault("dependencies", {})["lakehouse"] = {
            "default_lakehouse": lh_id, "default_lakehouse_name": lh_name, "default_lakehouse_workspace_id": ws_id}
    return {"format": "ipynb", "parts": [{"path": "notebook-content.ipynb", "payload": b64(json.dumps(nb)), "payloadType": "InlineBase64"}]}

def run_notebook(ws, item_id, stage, name, timeout_min=60, wait=True):
    r = call("POST", f"workspaces/{ws}/items/{item_id}/jobs/instances?jobType=RunNotebook", json={})
    if not wait:
        log(stage, f"run {name}", "started"); return
    loc, t0 = r.headers["Location"], time.time()
    while time.time() - t0 < timeout_min * 60:
        time.sleep(20)
        j = call("GET", loc).json(); st = j.get("status")
        if st in ("Completed", "Failed", "Cancelled", "Deduped"):
            if st != "Completed":
                raise RuntimeError(f"{name} {st}: {json.dumps(j.get('failureReason'))[:600]}")
            log(stage, f"run {name}", "completed", f"{int(time.time() - t0)} s"); return
    raise TimeoutError(f"{name} still running after {timeout_min} min")

# ---------- warehouse T-SQL ----------
def sql_token():
    for aud in ("https://database.windows.net/", "pbi"):
        try: return token(aud)
        except Exception: pass
    raise RuntimeError("Could not get a token for the warehouse")

def sql_connect(server, database):
    import pyodbc, struct
    drv = next((d for d in sorted(pyodbc.drivers(), reverse=True) if "ODBC Driver" in d and "SQL Server" in d), None)
    if not drv: raise RuntimeError(f"No SQL Server ODBC driver in this runtime: {pyodbc.drivers()}")
    tb = sql_token().encode("utf-16-le")
    return pyodbc.connect(f"Driver={{{drv}}};Server={server},1433;Database={database};Encrypt=Yes;TrustServerCertificate=No",
                          attrs_before={1256: struct.pack(f"<I{len(tb)}s", len(tb), tb)}, autocommit=True)

def run_batches(conn, script, show=False):
    for batch in re.split(r"^\s*GO\s*$", script, flags=re.M | re.I):
        if not batch.strip(): continue
        cur = conn.cursor(); cur.execute(batch)
        while True:
            if cur.description and show:
                cols = [d[0] for d in cur.description]
                display(pd.DataFrame.from_records([tuple(r) for r in cur.fetchall()], columns=cols))
            if not cur.nextset(): break
        cur.close()
print("Helpers ready. Workspace:", THIS_WS)

## 1 to 5. Core demo, one stage at a time

In [ ]:
def model_definition(server):
    base = "pbip/Monthly Member Intelligence Hub.SemanticModel/"
    parts = []
    for p in repo_list(base):
        rel = p[len(base):]
        if rel.startswith(".pbi/") or rel == ".platform": continue
        data = repo_bytes(p)
        if rel.endswith("expressions.tmdl"):
            t = data.decode("utf-8-sig")
            t = re.sub(r'(expression SqlEndpoint = ")[^"]*(")', rf'\g<1>{server}\g<2>', t)
            t = re.sub(r'(expression WarehouseName = ")[^"]*(")', rf'\g<1>{WAREHOUSE_NAME}\g<2>', t)
            data = t.encode("utf-8")
        parts.append({"path": rel, "payload": b64(data), "payloadType": "InlineBase64"})
    return {"format": "TMDL", "parts": parts}

def report_definition(model_id):
    base = "pbip/Monthly Member Intelligence Hub.Report/"
    parts = []
    for p in repo_list(base):
        rel = p[len(base):]
        if rel.startswith(".pbi/") or rel == ".platform": continue
        data = repo_bytes(p)
        if rel == "definition.pbir":
            pbir = json.loads(data.decode("utf-8-sig"))
            pbir["datasetReference"] = {"byConnection": {"connectionString": f"semanticmodelid={model_id}"}}
            data = json.dumps(pbir, indent=2).encode("utf-8")
        parts.append({"path": rel, "payload": b64(data), "payloadType": "InlineBase64"})
    return {"format": "PBIR", "parts": parts}

def set_model_credentials(ws, model_id, stage):
    ds = call("GET", f"groups/{ws}/datasets/{model_id}/datasources", base=PBI_API).json().get("value", [])
    if not ds:
        log(stage, "credentials", "skipped", "no data sources reported yet"); return
    cred = json.dumps({"credentialData": [{"name": "accessToken", "value": sql_token()}]})
    for d in ds:
        if not (d.get("gatewayId") and d.get("datasourceId")): continue
        call("PATCH", f"gateways/{d['gatewayId']}/datasources/{d['datasourceId']}", base=PBI_API, json={
            "credentialDetails": {"credentialType": "OAuth2", "credentials": cred, "encryptedConnection": "Encrypted",
                                  "encryptionAlgorithm": "None", "privacyLevel": "Organizational"}})
    log(stage, "credentials", "set (OAuth2)")

def refresh_model(ws, model_id, stage, timeout_min=20):
    call("POST", f"groups/{ws}/datasets/{model_id}/refreshes", base=PBI_API, json={"notifyOption": "NoNotification"})
    t0 = time.time()
    while time.time() - t0 < timeout_min * 60:
        time.sleep(15)
        h = call("GET", f"groups/{ws}/datasets/{model_id}/refreshes?$top=1", base=PBI_API).json()["value"][0]
        if h["status"] in ("Completed", "Failed", "Disabled", "Cancelled"):
            if h["status"] != "Completed":
                raise RuntimeError(f"Refresh {h['status']}: {h.get('serviceExceptionJson', '')[:500]}")
            log(stage, "model refresh", "completed"); return
    raise TimeoutError("Model refresh still running")

def persona_upns(stage_key):
    return {f"{p}@{TENANT_DOMAIN}".lower(): role for p, role in PERSONAS.items()
            if stage_key == "Dev" or p not in DEV_ONLY_PERSONAS}

def grant_workspace_access(ws, stage_key, stage):
    for upn, role in persona_upns(stage_key).items():
        r = call("POST", f"groups/{ws}/users", base=PBI_API, ok=None,
                 json={"emailAddress": upn, "groupUserAccessRight": role})
        if r.status_code in (200, 201): log(stage, f"access {upn}", role)
        elif "already" in r.text.lower(): log(stage, f"access {upn}", "already has access")
        else: log(stage, f"access {upn}", "FAILED", f"HTTP {r.status_code} {r.text[:150]} (does the account exist?)")

def add_rls_members(ws_name, stage_key, stage):
    srv = fabric.create_tom_server(readonly=False, workspace=ws_name)
    import Microsoft.AnalysisServices.Tabular as TOM
    db = srv.Databases.GetByName(MODEL_NAME)
    role = db.Model.Roles.Find(RLS_ROLE)
    have = {m.MemberName.lower() for m in role.Members}
    for upn in persona_upns(stage_key):
        if upn in have:
            log(stage, f"RLS {upn}", "already a member"); continue
        m = TOM.ExternalModelRoleMember(); m.MemberName = upn; m.IdentityProvider = "AzureAD"
        role.Members.Add(m)
        try:
            db.Model.SaveChanges(); log(stage, f"RLS {upn}", "added")
        except Exception as e:
            role.Members.Remove(m); db.Model.UndoLocalChanges() if hasattr(db.Model, "UndoLocalChanges") else None
            log(stage, f"RLS {upn}", "FAILED", e)
    srv.Disconnect()

def load_warehouse(conn, stage):
    run_batches(conn, repo_text("sql/warehouse/01_create_schema_and_tables.sql"))
    run_batches(conn, repo_text("sql/warehouse/02_usp_Load_Combined.sql"))
    for attempt in range(3):
        try:
            run_batches(conn, "EXEC [Combine-Data].[usp_Load_Combined];"); break
        except Exception as e:
            if attempt == 2: raise
            print("  load retry in 60 s (SQL endpoints still syncing):", str(e)[:200]); time.sleep(60)
    log(stage, "warehouse load", "completed")
    print("Validation:"); run_batches(conn, repo_text("sql/warehouse/03_validation_queries.sql"), show=True)
    conn.close()

def build_stage(stage_key, cap):
    stage = STAGES[stage_key]
    ws = ensure_workspace(stage, cap)

    # 2. Lakehouses and data
    load_params = {"TENANT_DOMAIN": TENANT_DOMAIN}
    if REPO_SOURCE == "github":
        load_params.update({"DATA_SOURCE": "github", "REPO_RAW_BASE": REPO_RAW_ROOT.rstrip("/") + "/data"})
    else:
        ref = ensure_item(ws, "Lakehouse", "Enterprise_Ref", stage=stage)
        files = wait_props(ws, f"lakehouses/{ref['id']}", "oneLakeFilesPath")["oneLakeFilesPath"]
        nbu.fs.cp(f"file:{FILES_ROOT.rstrip('/')}/data", f"{files.rstrip('/')}/data", True)
        load_params["DATA_SOURCE"] = "files"
    nb = ensure_item(ws, "Notebook", "Load_Source_Data", notebook_definition("notebooks/Load_Source_Data.ipynb", load_params), stage=stage)
    run_notebook(ws, nb["id"], stage, "Load_Source_Data")

    # 3. Warehouse
    wh = ensure_item(ws, "Warehouse", WAREHOUSE_NAME, stage=stage)
    server = wait_props(ws, f"warehouses/{wh['id']}", "connectionString")["connectionString"]
    try:
        conn = sql_connect(server, WAREHOUSE_NAME)
    except Exception as e:
        conn = None
        log(stage, "warehouse SQL", "SKIPPED", f"{e}. Run the three sql/warehouse scripts in the warehouse query editor, then rerun.")
    if conn is not None:
        load_warehouse(conn, stage)

    # 4. Semantic model and report
    model = ensure_item(ws, "SemanticModel", MODEL_NAME, model_definition(server), stage=stage)
    ensure_item(ws, "Report", REPORT_NAME, report_definition(model["id"]), stage=stage)
    if SET_CREDENTIALS:
        try: set_model_credentials(ws, model["id"], stage)
        except Exception as e: log(stage, "credentials", "FAILED", e)
    if REFRESH_MODEL:
        try: refresh_model(ws, model["id"], stage)
        except Exception as e: log(stage, "model refresh", "FAILED", e)

    # 5. Access and RLS
    if GRANT_WORKSPACE_ACCESS: grant_workspace_access(ws, stage_key, stage)
    if ADD_RLS_MEMBERS:
        try: add_rls_members(stage, stage_key, stage)
        except Exception as e: log(stage, "RLS members", "FAILED", e)
    return ws

STAGE_WS = {}
if BUILD_CORE:
    cap = capacity_id()
    for key in BUILD_STAGES:
        try:
            STAGE_WS[key] = build_stage(key, cap)
        except Exception as e:
            log(STAGES[key], "stage", "FAILED", e)

## 6. Deployment pipeline (optional)

In [ ]:
if CREATE_PIPELINE:
    if set(STAGE_WS) != {"Dev", "Test", "Prod"}:
        log("pipeline", "create", "skipped", "build all three stages first")
    else:
        try:
            pl = next((p for p in paged("deploymentPipelines") if p["displayName"] == PIPELINE_NAME), None)
            if pl is None:
                pl = call("POST", "deploymentPipelines", json={"displayName": PIPELINE_NAME, "stages": [
                    {"displayName": "Development", "isPublic": False},
                    {"displayName": "Test", "isPublic": False},
                    {"displayName": "Production", "isPublic": False}]}).json()
            stages = call("GET", f"deploymentPipelines/{pl['id']}/stages").json()["value"]
            for st, key in zip(sorted(stages, key=lambda s: s["order"]), ["Dev", "Test", "Prod"]):
                if st.get("workspaceId") != STAGE_WS[key]:
                    call("POST", f"deploymentPipelines/{pl['id']}/stages/{st['id']}/assignWorkspace",
                         json={"workspaceId": STAGE_WS[key]})
            log("pipeline", PIPELINE_NAME, "ready", "stages assigned; items pair by name and type")
        except Exception as e:
            log("pipeline", PIPELINE_NAME, "FAILED", e)

## 7. Real-Time Estate Monitor (this workspace)

In [ ]:
def put_kv_secret(value):
    r = requests.put(f"{KEY_VAULT_URI.rstrip('/')}/secrets/{EVENTSTREAM_SECRET_NAME}?api-version=7.4",
                     headers={"Authorization": f"Bearer {token('keyvault')}"}, json={"value": value}, timeout=60)
    r.raise_for_status()

def eventstream_definition(db):
    es = {
        "sources": [{"name": "collector", "type": "CustomEndpoint", "properties": {}}],
        "destinations": [{
            "name": "EstateMonitor", "type": "Eventhouse",
            "properties": {"dataIngestionMode": "ProcessedIngestion", "workspaceId": THIS_WS, "itemId": db["id"],
                           "databaseName": db["displayName"], "tableName": "RawEvents",
                           "inputSerialization": {"type": "Json", "properties": {"encoding": "UTF8"}}},
            "inputNodes": [{"name": f"{EVENTSTREAM_NAME}-stream"}]}],
        "streams": [{"name": f"{EVENTSTREAM_NAME}-stream", "type": "DefaultStream", "properties": {},
                     "inputNodes": [{"name": "collector"}]}],
        "operators": [], "compatibilityLevel": "1.0"}
    return {"parts": [{"path": "eventstream.json", "payload": b64(json.dumps(es)), "payloadType": "InlineBase64"}]}

def eventstream_connection_string(es_id):
    for _ in range(30):
        try:
            topo = call("GET", f"workspaces/{THIS_WS}/eventstreams/{es_id}/topology").json()
            src = next(s for s in topo.get("sources", []) if s["name"] == "collector")
            c = call("GET", f"workspaces/{THIS_WS}/eventstreams/{es_id}/sources/{src['id']}/connection").json()
            cs = (c.get("accessKeys") or {}).get("primaryConnectionString")
            if cs: return cs
        except Exception as e:
            last = e
        time.sleep(10)
    raise RuntimeError(f"Custom endpoint connection string not available: {last}")

if BUILD_ESTATE_MONITOR:
    M = "0-MONITOR"
    try:
        lh = ensure_item(THIS_WS, "Lakehouse", MONITOR_LAKEHOUSE, stage=M)
        LH = (THIS_WS, lh["id"], MONITOR_LAKEHOUSE)

        setup_params = {"EVENTHOUSE_NAME": EVENTHOUSE_NAME}
        if REPO_SOURCE == "github":
            setup_params.update({"SOURCE": "github", "REPO_RAW_ROOT": REPO_RAW_ROOT.rstrip("/")})
            setup_lh = None
        else:
            setup_params.update({"SOURCE": "files", "FILES_ROOT": FILES_ROOT}); setup_lh = LH
        nb = ensure_item(THIS_WS, "Notebook", "Estate_Monitor_Setup",
                         notebook_definition("notebooks/Estate_Monitor_Setup.ipynb", setup_params, setup_lh), stage=M)
        run_notebook(THIS_WS, nb["id"], M, "Estate_Monitor_Setup")

        eh = find_item(THIS_WS, "Eventhouse", EVENTHOUSE_NAME)
        db = next(d for d in paged(f"workspaces/{THIS_WS}/kqlDatabases")
                  if (d.get("properties") or {}).get("parentEventhouseItemId") == eh["id"])
        QUERY_URI = db["properties"]["queryServiceUri"].rstrip("/")

        sink = {"SINK": "eventhouse", "EVENTHOUSE_QUERY_URI": QUERY_URI, "KQL_DATABASE": db["displayName"]}
        if CREATE_EVENTSTREAM:
            try:
                es = ensure_item(THIS_WS, "Eventstream", EVENTSTREAM_NAME, eventstream_definition(db), update=False, stage=M)
                cs = eventstream_connection_string(es["id"])
                if KEY_VAULT_URI:
                    put_kv_secret(cs)
                    sink = {"SINK": "eventstream", "KEY_VAULT_URI": KEY_VAULT_URI, "EVENTSTREAM_SECRET_NAME": EVENTSTREAM_SECRET_NAME,
                            "EVENTHOUSE_QUERY_URI": QUERY_URI}
                    log(M, "eventstream secret", "stored in Key Vault")
                elif ALLOW_INLINE_CONNECTION_STRING:
                    sink = {"SINK": "eventstream", "EVENTSTREAM_CONNECTION_STRING": cs, "EVENTHOUSE_QUERY_URI": QUERY_URI}
                    log(M, "eventstream secret", "written into the collector and simulator notebooks")
                else:
                    log(M, "eventstream", "created", "collector writes straight to the eventhouse (no secret storage chosen)")
            except Exception as e:
                log(M, "eventstream", "FAILED", f"{e} -> collector writes straight to the eventhouse")

        col = ensure_item(THIS_WS, "Notebook", "Estate_Monitor_Collector",
                          notebook_definition("notebooks/Estate_Monitor_Collector.ipynb", sink, LH), stage=M)
        sim_params = dict(sink, TENANT_DOMAIN=TENANT_DOMAIN)
        ensure_item(THIS_WS, "Notebook", "Estate_Monitor_Simulator",
                    notebook_definition("notebooks/Estate_Monitor_Simulator.ipynb", sim_params, LH), stage=M)
        if IMPORT_GOVERNANCE:
            ensure_item(THIS_WS, "Notebook", "Fabric_Governance",
                        notebook_definition("notebooks/Fabric_Governance.ipynb", None, LH), stage=M)

        if SCHEDULE_COLLECTOR:
            sch = call("GET", f"workspaces/{THIS_WS}/items/{col['id']}/jobs/RunNotebook/schedules").json().get("value", [])
            if sch:
                log(M, "collector schedule", "exists")
            else:
                now = dt.datetime.utcnow().replace(microsecond=0)
                call("POST", f"workspaces/{THIS_WS}/items/{col['id']}/jobs/RunNotebook/schedules", json={
                    "enabled": True, "configuration": {
                        "type": "Cron", "interval": 60, "localTimeZoneId": "UTC",
                        "startDateTime": (now + dt.timedelta(minutes=60 if START_COLLECTOR_NOW else 5)).isoformat() + "Z",
                        "endDateTime": (now + dt.timedelta(days=365)).isoformat() + "Z"}})
                log(M, "collector schedule", "hourly")
        if START_COLLECTOR_NOW:
            run_notebook(THIS_WS, col["id"], M, "Estate_Monitor_Collector", wait=False)
    except Exception as e:
        log(M, "estate monitor", "FAILED", e)

## Summary and what's left by hand

In [ ]:
display(pd.DataFrame(LOG, columns=["Stage", "Step", "Status", "Detail"]))
failed = [l for l in LOG if l[2] == "FAILED"]
print(f"\n{len(failed)} step(s) failed." if failed else "\nEverything above completed.")

print("""
Still to do by hand (minutes each, see docs/01 and docs/07):
  1. Eventstream EstateMonitor_ES: add the Fabric event sources (Job, Workspace item, Capacity overview,
     Capacity operation events) and publish. The API can't subscribe to them yet.
  2. Semantic model in each stage: Settings > Data source credentials > Edit > OAuth2 and sign in once,
     so scheduled refresh keeps working (the installer's token is short-lived). Set endorsement to Certified.
  3. Tenant settings: Copilot and Azure OpenAI on for the demo users.
  4. App: create the org app or workspace app for the directors and the executive (doc 01, section 7).
  5. Optional: Activator alerts on the dashboard (doc 07, step 6) and a staged change in Dev for Compare (Act 2).
  6. Rehearse with docs/02 and the checklist in doc 01, section 10.
""")